# Prepare image data for image-image translation model training

## Overview
In this notebook, we will organize the image data into paired datasets for image-image translation model training, with the feature being the bright field channels and target being the 5 fluorescent channels, straitfying based on condition (cell line, seeding density, time point).  

## Import libraries

In [1]:
import os
import pathlib
import pandas as pd

In [36]:
# Location tothe extracted metadata information
METADATA_PATH = os.path.join(
    pathlib.Path(".").absolute(),
    "metadata"
)
TRAIN_INDEX_PATH = os.path.join(
    pathlib.Path(".").absolute(),
    "train_data_index.csv"
)

In [28]:
## load image index (PlateID, row, column, field, channel, file name and other image specific metadata)
filtered_metadata = pd.read_csv(os.path.join(METADATA_PATH, 'filtered_metadata.csv'), index_col=None)

# load re image status (PlateID)
channel_mappings = pd.read_csv(os.path.join(METADATA_PATH, 'channel_mappings.csv'), index_col=None)

In [29]:
filtered_metadata.head()

,Unnamed: 0,PlateID,MeasurementID,MeasurementStartTime,Version,id,State,URL,Row,Col,...,AbsTime,re_imaged,Date,Measurement,tiff_path,time_point,platemap_file,cell_line,well,seeding_density
0,0,BR00143978,57ddb76f-2d5b-4904-8743-bb6bafc8a296,2024-07-17T18:48:48.3580223-04:00,1,0314K1F1P1R1,Ok,r03c14f01p01-ch1sk1fk1fl1.tiff,C,14,...,2024-07-17T18:50:48.9006008-04:00,True,2024-07-17,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,48,Assay_Plate1_platemap,A673,C14,1000
1,49417,BR00143976,929737c3-51a4-43a5-a4f3-791872333f63,2024-07-04T16:04:45.222131-04:00,1,1210K1F5P1R4,Ok,r12c10f05p01-ch4sk1fk1fl1.tiff,L,10,...,2024-07-04T16:21:14.2476064-04:00,True,2024-07-17,5,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,24,Assay_Plate1_platemap,CHP212,L10,8000
2,49421,BR00143976,929737c3-51a4-43a5-a4f3-791872333f63,2024-07-04T16:04:45.222131-04:00,1,1210K1F5P1R6,Ok,r12c10f05p01-ch6sk1fk1fl1.tiff,L,10,...,2024-07-04T16:21:14.4972064-04:00,True,2024-07-17,5,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,24,Assay_Plate1_platemap,CHP212,L10,8000
3,49423,BR00143976,929737c3-51a4-43a5-a4f3-791872333f63,2024-07-04T16:04:45.222131-04:00,1,1210K1F6P1R1,Ok,r12c10f06p01-ch1sk1fk1fl1.tiff,L,10,...,2024-07-04T16:21:15.1368064-04:00,True,2024-07-17,5,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,24,Assay_Plate1_platemap,CHP212,L10,8000
4,49425,BR00143976,929737c3-51a4-43a5-a4f3-791872333f63,2024-07-04T16:04:45.222131-04:00,1,1210K1F6P1R2,Ok,r12c10f06p01-ch2sk1fk1fl1.tiff,L,10,...,2024-07-04T16:21:15.2616064-04:00,True,2024-07-17,5,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,24,Assay_Plate1_platemap,CHP212,L10,8000


In [30]:
channel_mappings.head()

,PlateID,channel_1,channel_2,channel_3,channel_4,channel_5,channel_6
0,BR00143978,Brightfield high,Alexa 488,Alexa 647,Alexa 568,Alexa 488 Long (CP),HOECHST 33342
1,BR00143980,Brightfield high,Alexa 488,Alexa 647,Alexa 568,Alexa 488 Long (CP),HOECHST 33342
2,BR00143976,Brightfield high,Alexa 488,Alexa 647,Alexa 568,Alexa 488 Long (CP),HOECHST 33342
3,BR00143977,Brightfield high,Alexa 488,Alexa 647,Alexa 568,Alexa 488 Long (CP),HOECHST 33342
4,BR00143981,Brightfield high,Alexa 488,Alexa 647,Alexa 568,Alexa 488 Long (CP),HOECHST 33342


In [35]:
# Define the target input channel and the fixed order of target channels
brightfield_channel = "Brightfield high"
target_channel_order = ["Alexa 488", "Alexa 647", "Alexa 568", "Alexa 488 Long (CP)", "HOECHST 33342"]

# List to store data rows
dataset_rows = []

# Group by the stratification variables
stratification_vars = ['PlateID', 'well', 'FieldID', 'Measurement']
grouped_metadata = filtered_metadata.groupby(stratification_vars)

# Process each group
for group_keys, group_data in grouped_metadata:
    plate_id = group_keys[0]
    
    # Get the channel mapping for this PlateID
    if plate_id not in channel_mappings["PlateID"].values:
        continue  # Skip if PlateID is not in channel_mappings
    channel_mapping = channel_mappings[channel_mappings["PlateID"] == plate_id].iloc[0]
    
    # Map ChannelID to channel names
    channel_id_to_name = {
        idx + 1: channel_mapping[f"channel_{idx + 1}"] 
        for idx in range(6)  # Assuming channel_1 to channel_6
    }
    
    # Filter for Brightfield channel
    brightfield_images = group_data[group_data["ChannelID"] == 1]  # Brightfield corresponds to ChannelID 1
    if brightfield_images.empty:
        continue  # Skip if no brightfield images in this group

    # Get paths to brightfield images
    brightfield_path = brightfield_images.iloc[0]["tiff_path"]  # Assuming one brightfield image per group
    
    # Collect paths for the target channels in the specified order
    target_paths = {}
    for target_channel_name in target_channel_order:
        # Find the corresponding ChannelID for the target channel name
        target_channel_id = next(
            (id for id, name in channel_id_to_name.items() if name == target_channel_name), None
        )
        if target_channel_id is not None:
            target_images = group_data[group_data["ChannelID"] == target_channel_id]
            if not target_images.empty:
                target_paths[target_channel_name] = target_images.iloc[0]["tiff_path"]
            else:
                target_paths[target_channel_name] = None  # If no image found for this channel
        else:
            target_paths[target_channel_name] = None  # If channel not defined in mapping
    
    # Ensure all target channels are present; otherwise, skip this group
    if all(path is not None for path in target_paths.values()):
        dataset_rows.append({
            "PlateID": plate_id,
            "well": group_keys[1],
            "FieldID": group_keys[2],
            "Measurement": group_keys[3],
            "Brightfield": brightfield_path,
            **{channel: target_paths[channel] for channel in target_channel_order}
        })

# Convert to DataFrame for saving and inspection
paired_data_df = pd.DataFrame(dataset_rows)

# Log final result
print(f"Paired dataset contains {len(paired_data_df)} rows.")
paired_data_df.head()


Paired dataset contains 14419 rows.


,PlateID,well,FieldID,Measurement,Brightfield,Alexa 488,Alexa 647,Alexa 568,Alexa 488 Long (CP),HOECHST 33342
0,BR00143976,C03,1,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...
1,BR00143976,C03,2,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...
2,BR00143976,C03,3,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...
3,BR00143976,C03,4,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...
4,BR00143976,C03,5,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...


In [37]:
paired_data_df.to_csv(TRAIN_INDEX_PATH, index=False)